# ch10 练习 —— 分组聚合

数据：`data/device_defects.csv`（270 行 x 13 列）

**每题都有 `assert` 验收块**。跑通即正确，跑不通的报错信息本身就说清了哪里错。
**不要翻答案版。**

| 题号 | 考什么 |
|---|---|
| TODO(1) | `size()` vs `count()`（差 21 行） |
| TODO(2) | 单函数聚合 + 均值/中位数对比 |
| TODO(3) | `agg` 函数列表 |
| TODO(4) | `agg` 字典（列名变 MultiIndex） |
| TODO(5) | **命名聚合**（含条件计数） |
| TODO(6) | `as_index=False` |
| TODO(7) | 多键分组 + `unstack` |
| TODO(8) | `dropna=False` |
| TODO(9) | `observed` 的真实默认值 |
| TODO(10) | 组内 top N（先排序再 head） |
| TODO(11) | `pivot_table` 与手写筛选取值对账 |
| TODO(12) | `transform` 组内 z-score |
| 综合 ① | 分组汇总表 + 柱状图 |
| 综合 ② | 分组统计的三条自检 |

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Arial Unicode MS", "STHeiti"]
plt.rcParams["axes.unicode_minus"] = False

DATA = Path("data")
dev = pd.read_csv(DATA / "device_defects.csv")

GROUP_KEY = "设备类型"
VALUE_COL = "负荷值"

print("pandas", pd.__version__)
print("dev", dev.shape)
print("分组基准：", dev.groupby(GROUP_KEY).size().to_dict())

---

## 基础题

In [ ]:
# 为 TODO(9) 准备：一个只有 2 类实际出现、但 category 记住全部 5 类的子集
full_dtype = pd.CategoricalDtype(categories=dev[GROUP_KEY].astype("category").cat.categories)
sub_dev = dev[dev[GROUP_KEY].isin(["变压器", "绝缘子"])]
print("full_dtype.categories:", list(full_dtype.categories))
print("sub_dev:", sub_dev.shape, "实际出现的类别:", list(sub_dev[GROUP_KEY].unique()))

In [ ]:
gb = dev.groupby(GROUP_KEY)

# TODO(1)：分别算出每组「行数」和「负荷值的有效个数」，存入 size_g 与 count_g
# 提示：gb.size() ；gb[VALUE_COL].count()
size_g = ____
count_g = ____
# ---- 验收 ----
assert size_g.to_dict() == {"互感器": 58, "变压器": 55, "断路器": 41, "绝缘子": 57, "避雷器": 59}
assert count_g.to_dict() == {"互感器": 55, "变压器": 49, "断路器": 37, "绝缘子": 53, "避雷器": 55}
assert int(size_g.sum()) == 270, "size 之和必须等于总行数"
assert int(count_g.sum()) == 249, f"count 之和应为 249（270-21），实际 {int(count_g.sum())}"
assert int(size_g.sum() - count_g.sum()) == 21, "差值必须等于负荷值缺失数"
print("TODO(1) 通过 -> size 合计", int(size_g.sum()), "| count 合计", int(count_g.sum()),
      "| 差", int(size_g.sum() - count_g.sum()))

In [ ]:
# TODO(2)：算出每组负荷值的均值与中位数，存入 mean_g 与 median_g
# 提示：dev.groupby(GROUP_KEY)[VALUE_COL].mean() ；... .median()
mean_g = ____
median_g = ____
# ---- 验收 ----
assert abs(mean_g["变压器"] - 986.9231) < 1e-3, f"变压器组均值应为 986.9231，实际 {mean_g['变压器']:.4f}"
assert abs(median_g["变压器"] - 521.08) < 0.01, f"变压器组中位数应为 521.08，实际 {median_g['变压器']:.2f}"
ratio = mean_g["变压器"] / median_g["变压器"]
assert ratio > 1.5, f"均值/中位数应明显大于 1（实际 {ratio:.2f}），否则说明没看出异常值污染"
assert len(mean_g) == 5 and mean_g.index.name == GROUP_KEY
print(f"TODO(2) 通过 -> 变压器组 mean={mean_g['变压器']:.2f} median={median_g['变压器']:.2f} "
      f"比值={ratio:.2f}（哨兵污染信号）")

---

## 进阶题

In [ ]:
# TODO(3)：用函数列表一次算出每组负荷值的 count/mean/median/std/max，存入 multi_stats
# 提示：dev.groupby(GROUP_KEY)[VALUE_COL].agg(["count","mean","median","std","max"])
multi_stats = ____
# ---- 验收 ----
assert multi_stats.shape == (5, 5), f"形状应为 (5, 5)，实际 {multi_stats.shape}"
assert list(multi_stats.columns) == ["count", "mean", "median", "std", "max"]
assert multi_stats.columns.nlevels == 1, "函数列表写法的列名应是单层"
assert multi_stats.loc["绝缘子", "count"] == 53
assert abs(multi_stats.loc["避雷器", "max"] - 103.81) < 0.01
print("TODO(3) 通过 -> 函数列表聚合", multi_stats.shape)
print(multi_stats.round(2).to_string())

In [ ]:
# TODO(4)：用字典写法：负荷值算 mean 与 max，湿度算 mean，存入 dict_stats
# 提示：dev.groupby(GROUP_KEY).agg({"负荷值": ["mean","max"], "湿度": "mean"})
dict_stats = ____
# ---- 验收 ----
assert dict_stats.shape == (5, 3), f"形状应为 (5, 3)，实际 {dict_stats.shape}"
assert dict_stats.columns.nlevels == 2, "字典写法的列名应是 MultiIndex（这就是它的坑）"
assert ("负荷值", "mean") in dict_stats.columns
assert ("湿度", "mean") in dict_stats.columns
assert abs(dict_stats[("负荷值", "mean")]["变压器"] - 986.9231) < 1e-3
# 验证压平成单层列名的正确做法
flat = dict_stats.copy()
flat.columns = ["_".join(col) for col in flat.columns]
assert list(flat.columns) == ["负荷值_mean", "负荷值_max", "湿度_mean"]
print("TODO(4) 通过 -> 字典聚合", dict_stats.shape, "（列名是元组，需手动压平）")

In [ ]:
# TODO(5)：用命名聚合产出一张指标表 summary，包含 5 个指标：
#         样本数(size)、有效数(count)、平均负荷(mean)、中位负荷(median)、危急数(条件计数)
# 提示：gb.agg(样本数=("负荷值","size"), ..., 危急数=("缺陷等级", lambda s: int((s=="危急").sum())))
summary = ____
# ---- 验收 ----
assert list(summary.columns) == ["样本数", "有效数", "平均负荷", "中位负荷", "危急数"], \
    f"列名不对：{list(summary.columns)}"
assert summary.columns.nlevels == 1, "命名聚合的列名应是单层"
assert summary.loc["互感器", "样本数"] == 58 and summary.loc["互感器", "有效数"] == 55
assert summary["危急数"].to_dict() == {"互感器": 5, "变压器": 5, "断路器": 4, "绝缘子": 4, "避雷器": 13}
assert abs(summary.loc["变压器", "平均负荷"] - 986.9231) < 1e-3
print("TODO(5) 通过 -> 命名聚合得到", list(summary.columns))
print(summary.round(2).to_string())

In [ ]:
# TODO(6)：用 as_index=False 重新产出 summary2，要求「设备类型」是普通列
# 提示：dev.groupby(GROUP_KEY, as_index=False).agg(样本数=("负荷值","size"), 平均负荷=("负荷值","mean"))
summary2 = ____
# ---- 验收 ----
assert list(summary2.columns) == [GROUP_KEY, "样本数", "平均负荷"], f"列名不对：{list(summary2.columns)}"
assert summary2.shape == (5, 3), f"形状应为 (5, 3)，实际 {summary2.shape}"
assert summary2[GROUP_KEY].tolist()[0] in {"互感器", "变压器", "断路器", "绝缘子", "避雷器"}
assert not isinstance(summary2.index, pd.MultiIndex) and summary2.index.tolist() == [0, 1, 2, 3, 4], \
    "as_index=False 后应是普通 RangeIndex"
print("TODO(6) 通过 -> 组键回到列里：", list(summary2.columns))

In [ ]:
# TODO(7)：按「台区编号 + 设备类型」双键分组求负荷均值，再 unstack 成交叉表，存入 cross
# 提示：dev.groupby(["台区编号", GROUP_KEY])[VALUE_COL].mean().unstack()
cross = ____
# ---- 验收 ----
assert cross.shape == (18, 5), f"交叉表应为 (18, 5)，实际 {cross.shape}"
assert list(cross.columns) == ["互感器", "变压器", "断路器", "绝缘子", "避雷器"]
assert list(cross.index.names) == ["台区编号"]
# 抽一格与手写筛选取值对账
cell = dev[(dev["台区编号"] == "STATION_A_01") & (dev[GROUP_KEY] == "断路器")][VALUE_COL].mean()
assert abs(cross.loc["STATION_A_01", "断路器"] - cell) < 1e-9, "交叉表取值与手写筛选不一致"
# STATION_A_01 没有互感器与变压器 -> 应为 NaN（该组合不存在）
assert pd.isna(cross.loc["STATION_A_01", "互感器"]), "不存在的组合应留 NaN，不是 0"
assert pd.isna(cross.loc["STATION_A_01", "变压器"])
print("TODO(7) 通过 -> 交叉表", cross.shape, "| A_01 行:", cross.loc["STATION_A_01"].round(1).to_dict())

In [ ]:
# TODO(8)：人为把前 5 行的缺陷等级置为缺失，然后分别用默认与 dropna=False 分组计数
#         存入 size_default 与 size_keepna
# 提示：先 d = dev.copy() 并置 NaN；再 d.groupby("缺陷等级").size() 与 d.groupby("缺陷等级", dropna=False).size()
d_na = ____
d_na.loc[0:4, "缺陷等级"] = ____
size_default = ____
size_keepna = ____
# ---- 验收 ----
assert size_default.to_dict() == {"一般": 161, "严重": 73, "危急": 31}, f"默认结果不对：{size_default.to_dict()}"
assert len(size_default) == 3
assert len(size_keepna) == 4, f"dropna=False 应有 4 组，实际 {len(size_keepna)}"
assert int(size_keepna.sum()) == 270, "dropna=False 的各组之和等于总行数"
assert int(size_default.sum()) == 265, f"默认丢掉了 5 行，实际求和 {int(size_default.sum())}"
assert int(size_keepna.sum() - size_default.sum()) == 5, "差值应正好是被丢掉的那 5 行"
print("TODO(8) 通过 -> 默认 3 组求和", int(size_default.sum()),
      "| dropna=False 4 组求和", int(size_keepna.sum()))

In [ ]:
# TODO(9)：验证 observed 的真实默认值：在「只有 2 类实际出现」的子集上分组
#         分别产出默认、observed=True、observed=False 三种结果
# 提示：先构造 full_dtype 并 astype，再 groupby(key).size() / groupby(key, observed=True).size() / (..., observed=False).size()
key_full = ____
size_obs_default = ____
size_obs_true = ____
size_obs_false = ____
# ---- 验收 ----
assert len(sub_dev) == 112, f"子集应有 112 行（变压器 55 + 绝缘子 57），实际 {len(sub_dev)}"
assert len(full_dtype.categories) == 5, "全量 category 应有 5 个类别"
assert len(size_obs_default) == 2, f"默认应为 2 组，实际 {len(size_obs_default)}"
assert len(size_obs_true) == 2
assert len(size_obs_false) == 5, f"observed=False 应产生 5 组（含 3 个空组），实际 {len(size_obs_false)}"
assert size_obs_default.to_dict() == size_obs_true.to_dict(), "默认值等价于 observed=True -> 说明 3.0 起默认已改为 True"
assert size_obs_false.to_dict() == {"互感器": 0, "变压器": 55, "断路器": 0, "绝缘子": 57, "避雷器": 0}
print("TODO(9) 通过 -> 默认", len(size_obs_default), "组 == observed=True；",
      "observed=False 才产生", len(size_obs_false), "组（3 个空组）")

In [ ]:
# TODO(10)：每组取负荷值最大的 1 条记录，存入 top1（先全局降序排序，再分组 head）
# 提示：dev.sort_values(VALUE_COL, ascending=False).groupby(GROUP_KEY).head(1)
top1 = ____
# ---- 验收 ----
assert top1.shape[0] == 5, f"每组 1 条，共 5 条，实际 {top1.shape[0]}"
assert len(top1[GROUP_KEY].unique()) == 5, "5 个组都要有代表"
worst = top1.set_index(GROUP_KEY)[VALUE_COL].to_dict()
assert worst["变压器"] == 9999.0 and worst["互感器"] == 9999.0 and worst["断路器"] == 9999.0, \
    "未清洗时三个组的 top1 都是哨兵值 9999"
assert top1.set_index(GROUP_KEY)["记录ID"].to_dict() == {
    "互感器": 91, "变压器": 16, "断路器": 112, "绝缘子": 190, "避雷器": 191,
}
# 清洗掉哨兵后重取，极值才真实
clean_sorted = dev[dev[VALUE_COL].between(0, 1000)].sort_values(VALUE_COL, ascending=False)
top1_clean = clean_sorted.groupby(GROUP_KEY).head(1)
assert top1_clean.set_index(GROUP_KEY)[VALUE_COL].to_dict()["绝缘子"] == 118.01
print("TODO(10) 通过 -> 未清洗 top1:", worst,
      "\n   清洗后 top1:", top1_clean.set_index(GROUP_KEY)[VALUE_COL].to_dict())

---

## 进阶题（续）

In [ ]:
# TODO(11)：用 pivot_table 做「设备类型 x 缺陷等级」的负荷均值交叉表，存入 pt
#         再单独取出「变压器 x 危急」这一格的值，存入 cell_manual（用手写筛选取均值）
# 提示：dev.pivot_table(index=GROUP_KEY, columns="缺陷等级", values=VALUE_COL, aggfunc="mean")
pt = ____
cell_manual = ____
# ---- 验收 ----
assert pt.shape == (5, 3), f"形状应为 (5, 3)，实际 {pt.shape}"
assert list(pt.columns) == ["一般", "严重", "危急"]
assert abs(pt.loc["变压器", "危急"] - 528.394) < 1e-3, f"该格应为 528.394，实际 {pt.loc['变压器', '危急']:.4f}"
assert abs(cell_manual - 528.394) < 1e-3
assert abs(pt.loc["变压器", "危急"] - cell_manual) < 1e-9, "pivot_table 与手写筛选必须一致"
# 与 groupby + unstack 写法完全等价
gt = dev.groupby([GROUP_KEY, "缺陷等级"])[VALUE_COL].mean().unstack()
assert pt.shape == gt.shape and bool(np.allclose(pt.to_numpy(), gt.to_numpy(), equal_nan=True)), \
    "pivot_table 应等价于 groupby + unstack"
# 默认 aggfunc 是 mean：不写 aggfunc 时结果应与上面一致
pt_default = dev.pivot_table(index=GROUP_KEY, columns="缺陷等级", values=VALUE_COL)
assert bool(np.allclose(pt_default.to_numpy(), pt.to_numpy(), equal_nan=True)), \
    "不写 aggfunc 时默认就是求均值（这就是那个暗坑）"
print("TODO(11) 通过 -> pivot_table", pt.shape, "| 变压器-危急 =", round(cell_manual, 4))

In [ ]:
# TODO(12)：用 transform 造三列特征：组均值、组内 z-score、组内百分位排名
#         存入 feat（一个含 记录ID / 设备类型 / 负荷值 / 组均值 / 组内z / 组内排名 的 DataFrame）
# 提示：feat = dev[["记录ID", GROUP_KEY, VALUE_COL]].copy() ；再依次赋三列，
#       均值和 std 用 .transform("mean") / .transform("std")，排名用 .rank(pct=True)
feat = ____
feat["组均值"] = ____
feat["组内z"] = ____
feat["组内排名"] = ____
# ---- 验收 ----
assert feat.shape == (270, 6), f"形状应为 (270, 6)，实际 {feat.shape}"
assert len(feat) == len(dev), "transform 保持行数不变，这是它与 agg 的根本区别"
assert list(feat.columns) == ["记录ID", GROUP_KEY, VALUE_COL, "组均值", "组内z", "组内排名"]
# 组均值应等于对应的组统计量
assert abs(feat.loc[0, "组均值"] - 355.4531) < 1e-3, f"第 0 行组均值应为 355.4531，实际 {feat.loc[0, '组均值']:.4f}"
for grp, sub in feat.groupby(GROUP_KEY):
    assert sub["组均值"].nunique() == 1, f"{grp} 组的组均值应该只有一个值"
# z-score 前 3 行（与讲解一致）
assert np.allclose(feat["组内z"].head(3).round(3).tolist(), [-0.179, -0.221, 0.218], atol=1e-3)
# 排名在 (0, 1] 区间
assert feat["组内排名"].min() > 0 and feat["组内排名"].max() == 1.0
# 组内排名的均值应约为 0.5
assert abs(feat.groupby(GROUP_KEY)["组内排名"].mean().mean() - 0.5) < 0.02
print("TODO(12) 通过 -> 造出 3 个组内特征，z 前 3 行:",
      feat["组内z"].head(3).round(3).tolist())

---

## 综合题

### 综合 ① 分组汇总表 + 柱状图

产出一张**可以直接交卷**的分组汇总表，并且画出「各设备类型的平均负荷 vs 中位负荷」对比柱状图。

两个要求：

1. 表要满足：`设备类型` 是普通列、行数 = 组数、含 5 个指标
2. 图要能一眼看出"均值被异常值拉高"——这是本章的核心结论

In [ ]:
# TODO(13)：综合①：产出 report（as_index=False 的汇总表）与柱状图 ax
# 提示：groupby(GROUP_KEY, as_index=False).agg(...) ；再 report.set_index(GROUP_KEY)[["平均负荷","中位负荷"]].plot.bar(ax=ax)
report = ____
fig, ax = ____
____
____
____
____
____
____
# ---- 验收 ----
assert list(report.columns) == [GROUP_KEY, "样本数", "有效数", "平均负荷", "中位负荷", "危急数"], \
    f"列名不对：{list(report.columns)}"
assert report.shape == (5, 6), f"形状应为 (5, 6)，实际 {report.shape}"
assert report[GROUP_KEY].notna().all(), "设备类型必须是普通列"
# 自检 1：各组样本数之和 == 总行数
assert int(report["样本数"].sum()) == 270, "分组漏了行（检查 dropna）"
# 自检 2：组数 == 键的 nunique
assert len(report) == dev[GROUP_KEY].nunique(), "组数与 nunique 不一致（检查 observed）"
# 自检 3：有效数之和 == 非缺失数
assert int(report["有效数"].sum()) == int(dev[VALUE_COL].notna().sum()) == 249
# 均值/中位数比：被哨兵污染的组比值明显 > 1，没被污染的组比值 ≈ 1
ratio = report["平均负荷"] / report["中位负荷"]
assert int((ratio > 1).sum()) >= 4, f"多数组应满足 均值 > 中位数，实际只有 {int((ratio > 1).sum())} 组"
assert ratio[report[GROUP_KEY] == "变压器"].iloc[0] > 1.5, "变压器组污染最重，比值应 > 1.5"
assert ratio[report[GROUP_KEY] == "避雷器"].iloc[0] < 1.05,     "避雷器组最大值仅 103.81、没有哨兵，均值应≈中位数（比值≈1）"
assert abs(report.loc[report[GROUP_KEY] == "变压器", "平均负荷"].iloc[0] - 986.9231) < 1e-3
# 图
assert ax is not None and len(ax.patches) >= 8, f"柱子数不对：{len(ax.patches)}"
print("综合① 通过 ->")
print(report.round(2).to_string(index=False))

### 综合 ② 分组统计的三条自检

把本章反复强调的三条自检写成一个可复用的检查结果：给定 `groupby` 结果与原始表，
输出三个布尔量。竞赛里这三行能省掉大量 debug 时间。

In [ ]:
# TODO(14)：综合②：对 report 与 gb 做三条自检，产出 checks 字典
# 提示：三个键：行数守恒 / 组数一致 / 组键在列；再补一个「哨兵是否污染极值」的键
n_rows_ok = ____
n_group_ok = ____
key_is_col = ____
sentinel_pollutes = ____
checks = ____
# ---- 验收 ----
expect = {"行数守恒", "组数一致", "组键在列", "哨兵污染极值"}
assert set(checks) == expect, f"键不对。多 {set(checks) - expect}，缺 {expect - set(checks)}"
assert checks["行数守恒"] is True
assert checks["组数一致"] is True
assert checks["组键在列"] is True
assert checks["哨兵污染极值"] is True, "未清洗时组内 top1 应被 9999 污染，这正是要检测出来的"
assert all(isinstance(v, bool) for v in checks.values()), "四项都应是布尔量"
print("综合② 通过 -> 分组统计自检：")
for key, value in checks.items():
    print(f"   {key:10s} = {value}")

---

### 复盘提问

**复盘问自己**（五题都能答上来才算过）：

1. `size()` 和 `count()` 差在哪？什么情况下两者相等、以至于你发现不了写错？
2. `agg` 四种写法里，只有一种能直接拿到可用的列名，是哪一种？为什么？
3. `groupby` 的 `dropna` 默认 `True` 会造成什么后果？写下那条能抓住它的自检。
4. 本机 pandas 3.0 的 `observed` 默认值是 `True` 还是 `False`？老教程为什么教反了？
5. `groupby().head(1)` 能不能拿到"负荷值最大的那条"？不能的话正确写法是什么？

答不上来的，回 `_notes/错题本.md` 记一笔。